# Retail BI: raw Excel to Power BI-ready star schema

This notebook is the complete, rerunnable data pipeline for the retail portfolio project. It starts from the original **Online Retail II** Excel workbook, profiles and cleans the source, uses **DuckDB SQL** to build an analytical star schema, validates every relationship, and exports compressed Parquet tables for Power BI.

The notebook deliberately keeps raw, cleaned, and analytical layers separate. It does not silently overwrite the source workbook.

## 1. Imports and project paths

The path function searches upward from the current working directory, so the notebook works whether it is launched from the repository root or from the `notebooks` folder.

In [1]:
from pathlib import Path
import json

import duckdb
import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "online_retail_2.xlsx").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw/online_retail_2.xlsx. "
        "Open this notebook from inside the retail-bi-project repository."
    )


ROOT = find_project_root()
RAW_XLSX = ROOT / "data" / "raw" / "online_retail_2.xlsx"
PROCESSED_DIR = ROOT / "data" / "processed"
VALIDATION_DIR = ROOT / "docs" / "validation"
SQL_FILE = ROOT / "sql" / "queries" / "01_build_retail_model.sql"
DB_FILE = PROCESSED_DIR / "retail_bi.duckdb"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Raw workbook: {RAW_XLSX}")
print(f"Workbook exists: {RAW_XLSX.exists()}")

Project root: <PROJECT_ROOT>
Raw workbook: <PROJECT_ROOT>\data\raw\online_retail_2.xlsx
Workbook exists: True


## 2. Load both raw workbook sheets

Each worksheet is loaded separately and receives two provenance fields:

- `source_period`: the worksheet/year from which the row came;
- `source_row`: the original row position inside that worksheet.

The combined table has one row per original workbook line. No cleaning happens in this step.

In [2]:
excel_file = pd.ExcelFile(RAW_XLSX)
print("Worksheets:", excel_file.sheet_names)

frames = []
for sheet_name in excel_file.sheet_names:
    frame = pd.read_excel(excel_file, sheet_name=sheet_name)
    frame.columns = (
        frame.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    frame = frame.rename(columns={
        "stockcode": "stock_code",
        "invoicedate": "invoice_date",
    })
    frame.insert(0, "source_row", range(2, len(frame) + 2))
    frame.insert(0, "source_period", sheet_name)
    frames.append(frame)
    print(f"{sheet_name}: {len(frame):,} data rows")

raw_df = pd.concat(frames, ignore_index=True)
print(f"Combined raw rows: {len(raw_df):,}")
print(f"Combined columns: {len(raw_df.columns):,}")
display(raw_df.head())

Worksheets: ['Year 2009-2010', 'Year 2010-2011']


Year 2009-2010: 525,461 data rows


Year 2010-2011: 541,910 data rows
Combined raw rows: 1,067,371
Combined columns: 10


,source_period,source_row,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
0,Year 2009-2010,2,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,Year 2009-2010,3,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,Year 2009-2010,4,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,Year 2009-2010,5,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,Year 2009-2010,6,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 3. Confirm the raw source structure

The workbook should contain 1,067,371 data rows: 525,461 in 2009–2010 and 541,910 in 2010–2011. The assertions stop the pipeline if a different or incomplete source file is used.

In [3]:
expected_rows_by_period = {
    "Year 2009-2010": 525_461,
    "Year 2010-2011": 541_910,
}

actual_rows_by_period = raw_df.groupby("source_period").size().to_dict()
expected_columns = {
    "source_period", "source_row", "invoice", "stock_code", "description",
    "quantity", "invoice_date", "price", "customer_id", "country"
}

assert actual_rows_by_period == expected_rows_by_period, (
    f"Unexpected worksheet counts: {actual_rows_by_period}"
)
assert set(raw_df.columns) == expected_columns, (
    f"Unexpected columns: {list(raw_df.columns)}"
)

raw_profile = pd.DataFrame({
    "metric": [
        "raw_rows", "raw_columns", "missing_description",
        "missing_customer_id", "distinct_invoices", "distinct_products",
        "minimum_invoice_datetime", "maximum_invoice_datetime"
    ],
    "value": [
        len(raw_df),
        len(raw_df.columns),
        int(raw_df["description"].isna().sum()),
        int(raw_df["customer_id"].isna().sum()),
        int(raw_df["invoice"].nunique(dropna=True)),
        int(raw_df["stock_code"].nunique(dropna=True)),
        str(raw_df["invoice_date"].min()),
        str(raw_df["invoice_date"].max()),
    ],
})
display(raw_profile)

,metric,value
0,raw_rows,1067371
1,raw_columns,10
2,missing_description,4382
3,missing_customer_id,243007
4,distinct_invoices,53628
5,distinct_products,5305
6,minimum_invoice_datetime,2009-12-01 07:45:00
7,maximum_invoice_datetime,2011-12-09 12:50:00


## 4. Load the raw layer into DuckDB

DuckDB can query a pandas DataFrame directly. The DataFrame is registered temporarily and then persisted as `raw_transactions` inside the local DuckDB database. The database and all generated data remain ignored by Git.

In [4]:
if DB_FILE.exists():
    DB_FILE.unlink()

con = duckdb.connect(str(DB_FILE))
con.register("raw_df", raw_df)
con.execute("CREATE TABLE raw_transactions AS SELECT * FROM raw_df")
con.unregister("raw_df")

raw_count = con.execute("SELECT COUNT(*) FROM raw_transactions").fetchone()[0]
assert raw_count == len(raw_df)
print(f"DuckDB raw rows: {raw_count:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB raw rows: 1,067,371


## 5. Audit cancellations, adjustments, and data quality

The source uses invoice prefixes as business signals. `C` invoices are cancellations/returns. `A` invoices are administrative adjustments. We measure these cases before deciding what enters the analytical fact table.

In [5]:
quality_profile = con.execute(
    '''
    SELECT
        COUNT(*) AS raw_rows,
        COUNT(*) FILTER (WHERE UPPER(CAST(invoice AS VARCHAR)) LIKE 'C%')
            AS cancellation_rows,
        COUNT(*) FILTER (WHERE UPPER(CAST(invoice AS VARCHAR)) LIKE 'A%')
            AS adjustment_rows,
        COUNT(*) FILTER (WHERE TRY_CAST(quantity AS INTEGER) < 0)
            AS negative_quantity_rows,
        COUNT(*) FILTER (WHERE TRY_CAST(price AS DOUBLE) <= 0)
            AS nonpositive_price_rows,
        COUNT(*) FILTER (WHERE customer_id IS NULL)
            AS missing_customer_rows,
        COUNT(*) FILTER (WHERE description IS NULL)
            AS missing_description_rows
    FROM raw_transactions
    '''
).df()
display(quality_profile)

,raw_rows,cancellation_rows,adjustment_rows,negative_quantity_rows,nonpositive_price_rows,missing_customer_rows,missing_description_rows
0,1067371,19494,6,22950,6207,243007,4382


In [6]:
duplicate_profile = con.execute(
    '''
    WITH grouped AS (
        SELECT
            invoice, stock_code, description, quantity, invoice_date,
            price, customer_id, country,
            COUNT(*) AS copies
        FROM raw_transactions
        GROUP BY ALL
    )
    SELECT
        SUM(copies - 1) AS exact_duplicate_rows,
        COUNT(*) FILTER (WHERE copies > 1) AS duplicated_row_patterns,
        MAX(copies) AS maximum_copies_of_one_row
    FROM grouped
    '''
).df()
display(duplicate_profile)

,exact_duplicate_rows,duplicated_row_patterns,maximum_copies_of_one_row
0,34335.0,32907,20


## 6. Build the cleaned layer and star schema with SQL

The SQL script performs the portfolio-visible modeling work:

1. standardizes types and blank values;
2. removes exact duplicated source lines;
3. retains valid positive-price sales and standard `C` returns;
4. creates the central transaction fact table;
5. creates date, product, customer, and country dimensions;
6. calculates customer RFM scores and segments;
7. creates monthly-sales and customer-cohort aggregates.

The original raw table remains unchanged.

In [7]:
MODEL_SQL = r"""-- Retail BI analytical model
-- Source table expected: raw_transactions
-- Grain of raw_transactions: one source workbook row.

CREATE OR REPLACE TABLE clean_transactions AS
SELECT
    source_period,
    CAST(source_row AS BIGINT) AS source_row,
    NULLIF(TRIM(CAST(invoice AS VARCHAR)), '') AS invoice_no,
    UPPER(NULLIF(TRIM(CAST(stock_code AS VARCHAR)), '')) AS stock_code,
    NULLIF(TRIM(CAST(description AS VARCHAR)), '') AS description,
    TRY_CAST(quantity AS INTEGER) AS quantity,
    TRY_CAST(invoice_date AS TIMESTAMP) AS invoice_datetime,
    TRY_CAST(price AS DOUBLE) AS unit_price,
    NULLIF(
        REGEXP_REPLACE(TRIM(CAST(customer_id AS VARCHAR)), '\\.0$', ''),
        ''
    ) AS customer_id,
    NULLIF(TRIM(CAST(country AS VARCHAR)), '') AS country
FROM raw_transactions;

CREATE OR REPLACE TABLE deduplicated_transactions AS
WITH numbered AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY
                invoice_no,
                stock_code,
                description,
                quantity,
                invoice_datetime,
                unit_price,
                customer_id,
                country
            ORDER BY source_period, source_row
        ) AS duplicate_number
    FROM clean_transactions
)
SELECT * EXCLUDE (duplicate_number)
FROM numbered
WHERE duplicate_number = 1;

CREATE OR REPLACE TABLE fact_transactions AS
SELECT
    'TXN:' || MD5(source_period || ':' || CAST(source_row AS VARCHAR))
        AS transaction_key,
    invoice_no,
    'PROD:' || stock_code AS product_key,
    CASE
        WHEN customer_id IS NULL THEN 'CUST:UNKNOWN'
        ELSE 'CUST:' || customer_id
    END AS customer_key,
    'COUNTRY:' || MD5(UPPER(country)) AS country_key,
    CAST(STRFTIME(invoice_datetime, '%Y%m%d') AS INTEGER) AS date_key,
    CAST(invoice_datetime AS DATE) AS invoice_date,
    invoice_datetime,
    CAST(invoice_datetime AS TIME) AS invoice_time,
    quantity,
    unit_price,
    quantity * unit_price AS line_value,
    CASE
        WHEN UPPER(invoice_no) LIKE 'C%' THEN 'Return'
        ELSE 'Sale'
    END AS transaction_type,
    CASE WHEN UPPER(invoice_no) LIKE 'C%' THEN TRUE ELSE FALSE END AS is_return,
    CASE WHEN UPPER(invoice_no) LIKE 'C%' THEN 0.0
         ELSE quantity * unit_price END AS gross_sales_value,
    CASE WHEN UPPER(invoice_no) LIKE 'C%' THEN ABS(quantity * unit_price)
         ELSE 0.0 END AS return_value,
    quantity * unit_price AS net_sales_value,
    source_period
FROM deduplicated_transactions
WHERE invoice_no IS NOT NULL
  AND stock_code IS NOT NULL
  AND country IS NOT NULL
  AND invoice_datetime IS NOT NULL
  AND unit_price > 0
  AND (
        (UPPER(invoice_no) NOT LIKE 'A%' AND UPPER(invoice_no) NOT LIKE 'C%' AND quantity > 0)
        OR
        (UPPER(invoice_no) LIKE 'C%' AND quantity < 0)
      );

CREATE OR REPLACE TABLE dim_product AS
WITH product_profile AS (
    SELECT
        stock_code,
        ARG_MAX(description, invoice_datetime)
            FILTER (WHERE description IS NOT NULL) AS product_description,
        MIN(CAST(invoice_datetime AS DATE)) AS first_observed_date,
        MAX(CAST(invoice_datetime AS DATE)) AS last_observed_date
    FROM deduplicated_transactions
    WHERE stock_code IS NOT NULL
    GROUP BY stock_code
)
SELECT
    'PROD:' || stock_code AS product_key,
    stock_code,
    COALESCE(product_description, 'Unknown product') AS product_description,
    first_observed_date,
    last_observed_date
FROM product_profile;

CREATE OR REPLACE TABLE dim_country AS
SELECT DISTINCT
    'COUNTRY:' || MD5(UPPER(country)) AS country_key,
    country
FROM deduplicated_transactions
WHERE country IS NOT NULL;

CREATE OR REPLACE TABLE dim_date AS
WITH bounds AS (
    SELECT MIN(invoice_date) AS min_date, MAX(invoice_date) AS max_date
    FROM fact_transactions
), calendar AS (
    SELECT CAST(calendar_date AS DATE) AS date
    FROM bounds,
    RANGE(min_date, max_date + INTERVAL 1 DAY, INTERVAL 1 DAY) AS t(calendar_date)
)
SELECT
    CAST(STRFTIME(date, '%Y%m%d') AS INTEGER) AS date_key,
    date,
    YEAR(date) AS year,
    QUARTER(date) AS quarter_number,
    'Q' || CAST(QUARTER(date) AS VARCHAR) AS quarter,
    MONTH(date) AS month_number,
    STRFTIME(date, '%B') AS month_name,
    STRFTIME(date, '%Y-%m') AS year_month,
    WEEK(date) AS week_number,
    DAYOFWEEK(date) AS day_of_week_number,
    STRFTIME(date, '%A') AS day_name,
    DAY(date) AS day_of_month,
    CASE WHEN DAYOFWEEK(date) IN (0, 6) THEN TRUE ELSE FALSE END AS is_weekend
FROM calendar;

CREATE OR REPLACE TABLE customer_metrics AS
WITH customer_base AS (
    SELECT
        customer_key,
        MIN(invoice_date) AS first_purchase_date,
        MAX(invoice_date) AS last_purchase_date,
        COUNT(DISTINCT CASE WHEN transaction_type = 'Sale' THEN invoice_no END)
            AS order_count,
        SUM(CASE WHEN transaction_type = 'Sale' THEN quantity ELSE 0 END)
            AS units_purchased,
        SUM(gross_sales_value) AS gross_sales_value,
        SUM(return_value) AS return_value,
        SUM(net_sales_value) AS net_sales_value
    FROM fact_transactions
    WHERE customer_key <> 'CUST:UNKNOWN'
    GROUP BY customer_key
), reference_date AS (
    SELECT MAX(invoice_date) + INTERVAL 1 DAY AS as_of_date
    FROM fact_transactions
), scored AS (
    SELECT
        cb.*,
        DATE_DIFF('day', last_purchase_date, rd.as_of_date) AS recency_days,
        6 - NTILE(5) OVER (ORDER BY DATE_DIFF('day', last_purchase_date, rd.as_of_date))
            AS recency_score,
        NTILE(5) OVER (ORDER BY order_count) AS frequency_score,
        NTILE(5) OVER (ORDER BY net_sales_value) AS monetary_score
    FROM customer_base cb
    CROSS JOIN reference_date rd
)
SELECT
    *,
    CAST(recency_score AS VARCHAR)
        || CAST(frequency_score AS VARCHAR)
        || CAST(monetary_score AS VARCHAR) AS rfm_code,
    CASE
        WHEN recency_score >= 4 AND frequency_score >= 4 AND monetary_score >= 4
            THEN 'Champions'
        WHEN recency_score >= 3 AND frequency_score >= 3
            THEN 'Loyal customers'
        WHEN recency_score >= 4 AND frequency_score <= 2
            THEN 'Promising'
        WHEN recency_score <= 2 AND frequency_score >= 4
            THEN 'At risk'
        WHEN recency_score <= 2 AND frequency_score <= 2
            THEN 'Hibernating'
        ELSE 'Needs attention'
    END AS rfm_segment
FROM scored;

CREATE OR REPLACE TABLE dim_customer AS
SELECT
    customer_key,
    REPLACE(customer_key, 'CUST:', '') AS customer_id,
    first_purchase_date,
    last_purchase_date,
    order_count,
    units_purchased,
    gross_sales_value,
    return_value,
    net_sales_value,
    recency_days,
    recency_score,
    frequency_score,
    monetary_score,
    rfm_code,
    rfm_segment
FROM customer_metrics
UNION ALL
SELECT
    'CUST:UNKNOWN',
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    NULL,
    'Unknown customer';

CREATE OR REPLACE TABLE agg_monthly_sales AS
SELECT
    YEAR(invoice_date) AS year,
    MONTH(invoice_date) AS month_number,
    STRFTIME(invoice_date, '%Y-%m') AS year_month,
    COUNT(DISTINCT CASE WHEN transaction_type = 'Sale' THEN invoice_no END)
        AS order_count,
    COUNT(DISTINCT CASE WHEN customer_key <> 'CUST:UNKNOWN' THEN customer_key END)
        AS identified_customer_count,
    SUM(CASE WHEN transaction_type = 'Sale' THEN quantity ELSE 0 END)
        AS units_sold,
    SUM(gross_sales_value) AS gross_sales_value,
    SUM(return_value) AS return_value,
    SUM(net_sales_value) AS net_sales_value
FROM fact_transactions
GROUP BY YEAR(invoice_date), MONTH(invoice_date), STRFTIME(invoice_date, '%Y-%m')
ORDER BY year, month_number;

CREATE OR REPLACE TABLE agg_customer_cohort AS
WITH first_purchase AS (
    SELECT customer_key, DATE_TRUNC('month', MIN(invoice_date)) AS cohort_month
    FROM fact_transactions
    WHERE customer_key <> 'CUST:UNKNOWN'
      AND transaction_type = 'Sale'
    GROUP BY customer_key
), activity AS (
    SELECT DISTINCT customer_key, DATE_TRUNC('month', invoice_date) AS activity_month
    FROM fact_transactions
    WHERE customer_key <> 'CUST:UNKNOWN'
      AND transaction_type = 'Sale'
)
SELECT
    fp.cohort_month,
    a.activity_month,
    DATE_DIFF('month', fp.cohort_month, a.activity_month) AS cohort_index,
    COUNT(DISTINCT a.customer_key) AS active_customers
FROM first_purchase fp
JOIN activity a USING (customer_key)
WHERE a.activity_month >= fp.cohort_month
GROUP BY fp.cohort_month, a.activity_month
ORDER BY fp.cohort_month, a.activity_month;

"""

# Prefer the repository SQL file when present so the notebook and GitHub SQL
# showcase always execute the same transformation logic.
if SQL_FILE.exists():
    MODEL_SQL = SQL_FILE.read_text(encoding="utf-8")

con.execute(MODEL_SQL)
print("Created cleaned tables, dimensions, fact table, and aggregates.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created cleaned tables, dimensions, fact table, and aggregates.


## 7. Reconcile row counts and transformation exclusions

This reconciliation makes the cleaning decisions explicit. The fact table includes:

- normal invoices with positive quantity and positive price;
- `C` invoices with negative quantity and positive price.

Administrative `A` rows, zero-price adjustments, malformed rows, and nonstandard negative-quantity rows are retained in the cleaned layer for audit but excluded from commercial KPIs.

In [8]:
reconciliation = con.execute(
    '''
    SELECT 'raw source rows' AS stage, COUNT(*) AS rows FROM raw_transactions
    UNION ALL
    SELECT 'clean typed rows', COUNT(*) FROM clean_transactions
    UNION ALL
    SELECT 'after exact deduplication', COUNT(*) FROM deduplicated_transactions
    UNION ALL
    SELECT 'analytical fact rows', COUNT(*) FROM fact_transactions
    '''
).df()
display(reconciliation)

exclusion_reasons = con.execute(
    '''
    SELECT
        CASE
            WHEN invoice_no IS NULL THEN 'Missing invoice'
            WHEN stock_code IS NULL THEN 'Missing stock code'
            WHEN country IS NULL THEN 'Missing country'
            WHEN invoice_datetime IS NULL THEN 'Invalid date'
            WHEN unit_price IS NULL OR unit_price <= 0 THEN 'Nonpositive/invalid price'
            WHEN UPPER(invoice_no) LIKE 'A%' THEN 'Administrative adjustment'
            WHEN UPPER(invoice_no) LIKE 'C%' AND quantity >= 0 THEN 'Nonstandard C row'
            WHEN UPPER(invoice_no) NOT LIKE 'C%' AND quantity <= 0 THEN 'Nonstandard quantity'
            ELSE 'Included'
        END AS disposition,
        COUNT(*) AS rows
    FROM deduplicated_transactions
    GROUP BY 1
    ORDER BY rows DESC
    '''
).df()
display(exclusion_reasons)

,stage,rows
0,raw source rows,1067371
1,clean typed rows,1067371
2,after exact deduplication,1033036
3,analytical fact rows,1027015


,disposition,rows
0,Included,1027015
1,Nonpositive/invalid price,6019
2,Administrative adjustment,1
3,Nonstandard C row,1


## 8. Validate star-schema keys and relationships

Every dimension key must be unique. Every fact foreign key must find one matching dimension row. These assertions intentionally stop the notebook before export if the semantic model is structurally unsafe.

In [9]:
dimension_checks = con.execute(
    '''
    SELECT 'dim_date' AS table_name, COUNT(*) AS rows,
           COUNT(DISTINCT date_key) AS distinct_keys FROM dim_date
    UNION ALL
    SELECT 'dim_product', COUNT(*), COUNT(DISTINCT product_key) FROM dim_product
    UNION ALL
    SELECT 'dim_customer', COUNT(*), COUNT(DISTINCT customer_key) FROM dim_customer
    UNION ALL
    SELECT 'dim_country', COUNT(*), COUNT(DISTINCT country_key) FROM dim_country
    '''
).df()

foreign_key_checks = con.execute(
    '''
    SELECT
        COUNT(*) FILTER (WHERE d.date_key IS NULL) AS unmatched_dates,
        COUNT(*) FILTER (WHERE p.product_key IS NULL) AS unmatched_products,
        COUNT(*) FILTER (WHERE c.customer_key IS NULL) AS unmatched_customers,
        COUNT(*) FILTER (WHERE g.country_key IS NULL) AS unmatched_countries
    FROM fact_transactions f
    LEFT JOIN dim_date d ON f.date_key = d.date_key
    LEFT JOIN dim_product p ON f.product_key = p.product_key
    LEFT JOIN dim_customer c ON f.customer_key = c.customer_key
    LEFT JOIN dim_country g ON f.country_key = g.country_key
    '''
).df()

display(dimension_checks)
display(foreign_key_checks)

assert (dimension_checks["rows"] == dimension_checks["distinct_keys"]).all(), (
    "At least one dimension has duplicate keys."
)
assert (foreign_key_checks.iloc[0] == 0).all(), (
    "At least one fact foreign key has no matching dimension row."
)
assert con.execute(
    "SELECT COUNT(*) = COUNT(DISTINCT transaction_key) FROM fact_transactions"
).fetchone()[0], "Transaction keys are not unique."

print("All star-schema validation checks passed.")

,table_name,rows,distinct_keys
0,dim_date,739,739
1,dim_product,5131,5131
2,dim_customer,5940,5940
3,dim_country,43,43


,unmatched_dates,unmatched_products,unmatched_customers,unmatched_countries
0,0,0,0,0


All star-schema validation checks passed.


## 9. Inspect the analytical results

These queries are sanity checks and examples of SQL-driven business analysis before Power BI.

In [10]:
kpi_summary = con.execute(
    '''
    SELECT
        COUNT(*) AS fact_rows,
        COUNT(DISTINCT CASE WHEN transaction_type = 'Sale' THEN invoice_no END)
            AS sales_orders,
        COUNT(DISTINCT CASE WHEN customer_key <> 'CUST:UNKNOWN' THEN customer_key END)
            AS identified_customers,
        COUNT(DISTINCT product_key) AS products,
        COUNT(DISTINCT country_key) AS countries,
        SUM(gross_sales_value) AS gross_sales,
        SUM(return_value) AS return_value,
        SUM(net_sales_value) AS net_sales
    FROM fact_transactions
    '''
).df()
display(kpi_summary)

,fact_rows,sales_orders,identified_customers,products,countries,gross_sales,return_value,net_sales
0,1027015,40076,5939,4759,43,2.046520e+07,1462424.18,1.900277e+07


In [11]:
top_products = con.execute(
    '''
    SELECT
        p.stock_code,
        p.product_description,
        SUM(f.net_sales_value) AS net_sales,
        COUNT(DISTINCT f.invoice_no) AS invoice_count
    FROM fact_transactions f
    JOIN dim_product p USING (product_key)
    GROUP BY p.stock_code, p.product_description
    ORDER BY net_sales DESC
    LIMIT 10
    '''
).df()
display(top_products)

,stock_code,product_description,net_sales,invoice_count
0,22423,REGENCY CAKESTAND 3 TIER,314045.02,4259
1,DOT,DOTCOM POSTAGE,309844.10,1418
2,85123A,CREAM HANGING HEART T-LIGHT HOLDER,251781.63,5598
3,85099B,JUMBO BAG RED RETROSPOT,180512.16,4096
4,47566,PARTY BUNTING,147079.73,2697
5,84879,ASSORTED COLOUR BIRD ORNAMENT,128550.42,2826
6,22086,PAPER CHAIN KIT 50'S CHRISTMAS,116298.59,2034
7,POST,POSTAGE,110430.41,2077
8,79321,CHILLI LIGHTS,79905.13,1156
9,22197,POPCORN HOLDER,78903.62,2412


In [12]:
rfm_summary = con.execute(
    '''
    SELECT
        rfm_segment,
        COUNT(*) AS customers,
        SUM(net_sales_value) AS customer_value,
        AVG(order_count) AS average_orders
    FROM dim_customer
    WHERE customer_key <> 'CUST:UNKNOWN'
    GROUP BY rfm_segment
    ORDER BY customer_value DESC
    '''
).df()
display(rfm_summary)

,rfm_segment,customers,customer_value,average_orders
0,Champions,1344,1.152997e+07,16.633185
1,Loyal customers,1521,2.597898e+06,5.182774
2,At risk,285,8.280057e+05,8.189474
3,Needs attention,829,6.557393e+05,2.381182
4,Hibernating,1677,5.681753e+05,1.251044
5,Promising,283,1.101990e+05,1.148410


## 10. Export Power BI-ready Parquet tables

Parquet preserves data types, compresses efficiently, and loads faster than CSV for this model. The exports remain local because `data/` is ignored by Git.

In [13]:
export_tables = [
    "fact_transactions",
    "dim_date",
    "dim_product",
    "dim_customer",
    "dim_country",
    "agg_monthly_sales",
    "agg_customer_cohort",
]

for table in export_tables:
    output_path = (PROCESSED_DIR / f"{table}.parquet").as_posix().replace("'", "''")
    con.execute(
        f"COPY {table} TO '{output_path}' "
        "(FORMAT PARQUET, COMPRESSION ZSTD, OVERWRITE_OR_IGNORE TRUE)"
    )
    print(f"Exported {table}.parquet")

Exported fact_transactions.parquet
Exported dim_date.parquet
Exported dim_product.parquet
Exported dim_customer.parquet
Exported dim_country.parquet
Exported agg_monthly_sales.parquet
Exported agg_customer_cohort.parquet


## 11. Create validation artifacts and export manifest

The manifest records table row counts and file sizes. It can be committed because it contains only metadata, not retail records.

In [14]:
manifest_rows = []
for table in export_tables:
    file_path = PROCESSED_DIR / f"{table}.parquet"
    manifest_rows.append({
        "table_name": table,
        "row_count": con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0],
        "file_name": file_path.name,
        "file_size_mb": round(file_path.stat().st_size / (1024 ** 2), 3),
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(VALIDATION_DIR / "power_bi_export_manifest.csv", index=False)
raw_profile.to_csv(VALIDATION_DIR / "raw_profile.csv", index=False)
reconciliation.to_csv(VALIDATION_DIR / "row_reconciliation.csv", index=False)
exclusion_reasons.to_csv(VALIDATION_DIR / "exclusion_reasons.csv", index=False)
dimension_checks.to_csv(VALIDATION_DIR / "dimension_key_checks.csv", index=False)
foreign_key_checks.to_csv(VALIDATION_DIR / "foreign_key_checks.csv", index=False)

display(manifest)

,table_name,row_count,file_name,file_size_mb
0,fact_transactions,1027015,fact_transactions.parquet,30.528
1,dim_date,739,dim_date.parquet,0.007
2,dim_product,5131,dim_product.parquet,0.093
3,dim_customer,5940,dim_customer.parquet,0.138
4,dim_country,43,dim_country.parquet,0.002
5,agg_monthly_sales,25,agg_monthly_sales.parquet,0.002
6,agg_customer_cohort,325,agg_customer_cohort.parquet,0.002


## 12. Completion gate

The pipeline is complete only when every exported file exists, each file has rows, and all structural assertions have passed.

In [15]:
assert all((PROCESSED_DIR / f"{table}.parquet").exists() for table in export_tables)
assert (manifest["row_count"] > 0).all()

con.close()

print("Retail BI data pipeline completed successfully.")
print(f"Power BI files: {PROCESSED_DIR}")
print(f"Validation reports: {VALIDATION_DIR}")

Retail BI data pipeline completed successfully.
Power BI files: <PROJECT_ROOT>\data\processed
Validation reports: <PROJECT_ROOT>\docs\validation


## Final modeling conclusion

- **Raw grain:** one line from one source worksheet.
- **Analytical fact grain:** one deduplicated, commercially valid invoice–product line.
- **Sales:** normal invoices with positive quantity and price.
- **Returns:** `C` invoices with negative quantity and positive price.
- **Administrative/stock adjustments:** retained in the audit layer but excluded from commercial KPIs.
- **Unknown customers:** retained under `CUST:UNKNOWN` so total sales reconcile while customer-specific analysis remains honest.
- **Power BI model:** one central transaction fact filtered by date, product, customer, and country dimensions.